# 🧪 Smrti Comprehensive Testing

**Goal**: Test all currently implemented functionality in Smrti to understand what works.

**Approach**: 
- Test low-level tier access directly
- Verify each storage backend
- Document what works and what doesn't
- Fix issues as we discover them

**Log**: All findings are documented in `LEARNING.md`

## 🔧 Setup & Initialization

### Prerequisites

Before running this notebook, ensure you have:

1. **Redis** running locally:
   ```bash
   # Install: sudo apt install redis-server
   # Or with Docker: docker run -d -p 6379:6379 redis
   ```

2. **ChromaDB** installed:
   ```bash
   pip install chromadb
   ```

3. **PostgreSQL** running (optional for episodic memory):
   ```bash
   # Or skip episodic memory testing for now
   ```

**Note**: The notebook will gracefully handle missing backends and test what's available.

In [1]:
# Import required modules
import asyncio
from datetime import datetime, timedelta
from typing import Dict, List, Optional
import json

# Import Smrti
from smrti import Smrti, SmrtiConfig
from smrti.schemas.models import RecordEnvelope

print("✅ Imports successful")

/home/bert/Work/orgs/konf-dev/smrti/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful


In [2]:
# Configure Smrti with all backend settings
config = SmrtiConfig(
    default_tenant_id="test",
    
    # Redis configuration (for Working & Short-Term Memory)
    redis_config={
        "host": "localhost",
        "port": 6379,
        "db": 0,
        "decode_responses": False
    },
    
    # ChromaDB configuration (for Long-Term Memory) - FIXED PORT
    chroma_config={
        "host": "localhost",
        "port": 8001,  # ChromaDB is running on port 8001
        "persist_directory": "./chroma_data"
    },
    
    # PostgreSQL configuration (for Episodic Memory) - FIXED CREDENTIALS
    postgres_config={
        "host": "localhost",
        "port": 5432,
        "database": "smrti",
        "user": "smrti",  # Correct user
        "password": "smrti_password"  # Correct password
    },
    
    # Sentence Transformers configuration
    sentence_transformers_config={
        "model_name": "all-MiniLM-L6-v2"
    },
    
    log_level="INFO"
)

print(f"✅ Config created")
print(f"   Tenant ID: {config.default_tenant_id}")
print(f"   Embedding Model: {config.sentence_transformers_config.get('model_name')}")
print(f"   Redis: {config.redis_config['host']}:{config.redis_config['port']}")
print(f"   ChromaDB: {config.chroma_config.get('host', 'localhost')}:{config.chroma_config.get('port', 8001)}")
print(f"   PostgreSQL: {config.postgres_config['user']}@{config.postgres_config['host']}:{config.postgres_config['port']}/{config.postgres_config['database']}")
print(f"   Log Level: {config.log_level}")

✅ Config created
   Tenant ID: test
   Embedding Model: all-MiniLM-L6-v2
   Redis: localhost:6379
   ChromaDB: localhost:8001
   PostgreSQL: smrti@localhost:5432/smrti
   Log Level: INFO


In [3]:
# Initialize Smrti
smrti = Smrti(config=config)
await smrti.initialize()

print("✅ Smrti initialized")
health = await smrti.get_health_status()
print(f"   Status: {health}")

INFO:smrti.adapters.smrti_main:Initializing Smrti intelligent memory system...
INFO:smrti.adapters.smrti_main:Initializing embedding providers...
INFO:smrti.adapters.smrti_main:Registered SentenceTransformers embedding provider
INFO:smrti.adapters.smrti_main:Initializing storage adapters...
INFO:smrti.adapters.smrti_main:Registered Redis adapter
INFO:smrti.adapters.smrti_main:Registered ChromaDB adapter
INFO:smrti.adapters.smrti_main:Registered PostgreSQL adapter
INFO:smrti.adapters.smrti_main:Initializing memory tiers...
INFO:smrti.adapters.smrti_main:Initializing core engines...
INFO:smrti.adapters.smrti_main:Initialized Context Assembly Engine
INFO:smrti.adapters.smrti_main:Initialized Unified Retrieval Engine
INFO:smrti.adapters.smrti_main:Initialized Memory Consolidation Engine
INFO:smrti.adapters.memory_consolidation:Memory consolidation service started
INFO:smrti.adapters.smrti_main:Smrti system initialized successfully in 0.07s
ERROR:smrti.adapters.unified_retrieval:Adapter uni

✅ Smrti initialized
   Status: {'status': 'degraded', 'timestamp': '2025-10-04T18:22:13.785989', 'components': {'retrieval_engine': {'status': 'unhealthy', 'error': "Health check failed for adapter unified_retrieval: 4 validation errors for MemoryQuery\nuser_id\n  Field required [type=missing, input_value={'query_text': 'test', 'l...ilarity_threshold': 0.9}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.11/v/missing\nquery\n  Field required [type=missing, input_value={'query_text': 'test', 'l...ilarity_threshold': 0.9}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.11/v/missing\ntenant\n  Field required [type=missing, input_value={'query_text': 'test', 'l...ilarity_threshold': 0.9}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.11/v/missing\nnamespace\n  Field required [type=missing, input_value={'query_text': 'test', 'l...ilarity_threshold': 0.9}, input_type=dict]\n    For furth

In [4]:
# Check system stats
stats = await smrti.get_system_stats()
print("📊 System Stats:")
print(json.dumps(stats, indent=2))

📊 System Stats:
{
  "system": {
    "initialized": true,
    "running": true,
    "initialization_time": "2025-10-04T18:22:13.715129",
    "total_operations": 0,
    "successful_operations": 0,
    "failed_operations": 0,
    "success_rate": 0.0,
    "active_sessions": 0
  },
  "registry": {
    "tier_stores": 3,
    "memory_tiers": 0,
    "embedding_providers": 1,
    "available_tiers": [],
    "available_embedders": [
      "sentence_transformers"
    ]
  },
  "retrieval_engine": {
    "error": "'RetrievalConfig' object has no attribute 'items'"
  },
  "context_assembly": {
    "error": "'ContextAssemblyConfig' object has no attribute 'items'"
  },
  "consolidation": {
    "error": "'ConsolidationConfig' object has no attribute 'items'"
  }
}


In [5]:
# Check what adapters are actually registered
print("📋 Registered Adapters:")
print(f"   Tier Stores: {list(smrti.registry.tier_stores.keys())}")
print(f"   Embedding Providers: {list(smrti.registry.embedding_providers.keys())}")

# Test direct adapter access
if 'redis' in smrti.registry.tier_stores:
    redis_adapter = smrti.registry.tier_stores['redis']
    print(f"\n✅ Redis adapter accessible: {redis_adapter}")
    
if 'chroma' in smrti.registry.tier_stores:
    chroma_adapter = smrti.registry.tier_stores['chroma']
    print(f"✅ ChromaDB adapter accessible: {chroma_adapter}")
    
if 'postgres' in smrti.registry.tier_stores:
    postgres_adapter = smrti.registry.tier_stores['postgres']
    print(f"✅ PostgreSQL adapter accessible: {postgres_adapter}")

📋 Registered Adapters:
   Tier Stores: ['redis', 'chroma', 'postgres']
   Embedding Providers: ['sentence_transformers']

✅ Redis adapter accessible: <smrti.adapters.storage.redis.RedisAdapter object at 0x7f48092dcad0>
✅ ChromaDB adapter accessible: <smrti.adapters.vector.chromadb.ChromaDBAdapter object at 0x7f48092dcc20>
✅ PostgreSQL adapter accessible: <smrti.adapters.database.postgresql.PostgreSQLAdapter object at 0x7f48092dcd70>


In [6]:
# Test 1: Redis Adapter - Store and Retrieve  
print("🧪 Testing Redis Adapter (Low-Level API)\n")

# Import the fixed RedisAdapter class
from smrti.adapters.storage.redis import RedisAdapter

# Create a NEW instance with proper config (bypassing the registry)
redis_adapter_new = RedisAdapter(
    tier_name="redis_test",
    config={
        "host": "localhost",
        "port": 6379,
        "db": 0
    }
)

# Initialize the adapter (establish connection)
print("Initializing Redis adapter...")
await redis_adapter_new._initialize_connections()
print("✅ Redis connection established\n")

# Create a test record (ID is auto-generated)
test_record = RecordEnvelope(
    tenant="test",
    namespace="test_user_123",
    user_id="user_123",
    tier="working",
    content_type="TEXT",
    payload="Testing Redis adapter direct storage",
    importance_score=0.8,
    metadata={"test": "redis_direct"},
    created_at=datetime.utcnow()
)

print(f"Created record with ID: {test_record.id}")
print(f"  record.record_id (alias) = {test_record.record_id}\n")

# Store the record
try:
    record_id = await redis_adapter_new.store(test_record)
    print(f"✅ Stored in Redis: {record_id}")
    
    # Retrieve it back using the returned ID
    retrieved = await redis_adapter_new.retrieve(record_id)
    print(f"✅ Retrieved from Redis")
    print(f"   Payload: {retrieved.payload if retrieved else 'NOT FOUND'}")
    print(f"   Match: {retrieved.payload == test_record.payload if retrieved else False}")
    
except Exception as e:
    print(f"❌ Redis test failed: {e}")
    import traceback
    traceback.print_exc()

INFO:smrti.adapters.redis_test:Redis adapter redis_test initialized (host=localhost:6379, db=0, replicas=0)


🧪 Testing Redis Adapter (Low-Level API)

Initializing Redis adapter...
✅ Redis connection established

Created record with ID: 6ffb463e-318f-47dc-bd0e-77db23afc66d
  record.record_id (alias) = 6ffb463e-318f-47dc-bd0e-77db23afc66d

✅ Stored in Redis: 6ffb463e-318f-47dc-bd0e-77db23afc66d
✅ Retrieved from Redis
   Payload: Testing Redis adapter direct storage
   Match: True


/tmp/ipykernel_1377587/3110405854.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow()


## ✅ Field Name Inconsistency Fixed!

**Root Cause:** The entire codebase was using `record.record_id` and `record.tenant_id`, but the `RecordEnvelope` model defined these fields as `record.id` and `record.tenant`.

**Solution Applied:** Added property aliases to `RecordEnvelope`:
```python
@property
def record_id(self) -> UUID:
    """Alias for id field (backward compatibility)."""
    return self.id

@property
def tenant_id(self) -> str:
    """Alias for tenant field (backward compatibility)."""
    return self.tenant
```

Now both field names work! Please **restart the kernel** and re-run cells 1-5 to test.

In [7]:
# Force reload of the models and adapter modules
import importlib
import sys
from datetime import datetime

# Remove cached modules
modules_to_reload = [
    'smrti.schemas.models',
    'smrti.core.base',
    'smrti.adapters.storage.redis'
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]

# Re-import to get fresh code
from smrti.schemas.models import RecordEnvelope

# Test the aliases work
test_record = RecordEnvelope(
    tenant="test",
    namespace="testing", 
    user_id="test_user",
    tier="working",
    content_type="TEXT",
    payload="Testing aliases",
    created_at=datetime.utcnow()
)

print(f"✅ record.id = {test_record.id}")
print(f"✅ record.record_id (alias) = {test_record.record_id}")
print(f"✅ record.tenant = {test_record.tenant}")
print(f"✅ record.tenant_id (alias) = {test_record.tenant_id}")
print("\n🔄 Modules reloaded! Redis adapter SSL fix applied.")

✅ record.id = c9a75132-5687-46db-bd53-a6b35cc505c4
✅ record.record_id (alias) = c9a75132-5687-46db-bd53-a6b35cc505c4
✅ record.tenant = test
✅ record.tenant_id (alias) = test

🔄 Modules reloaded! Redis adapter SSL fix applied.


/tmp/ipykernel_1377587/1455184298.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow()


## 🧠 Test 1: Working Memory (Redis with TTL)

Working memory should:
- Store items temporarily (5 minute default TTL)
- Use Redis as backend
- Handle recent/active information

In [8]:
# Create a test memory record
working_memory_record = RecordEnvelope(
    tenant="test",
    namespace="test_user_123",
    user_id="user_123",
    tier="working",
    content_type="TEXT",
    payload="User is currently working on testing Smrti's memory tiers",
    importance_score=0.8,
    metadata={
        "type": "current_activity",
        "source": "test_notebook"
    },
    created_at=datetime.utcnow()
)

print("📝 Created working memory record:")
print(f"   Content: {working_memory_record.payload}")
print(f"   Metadata: {working_memory_record.metadata}")

📝 Created working memory record:
   Content: User is currently working on testing Smrti's memory tiers
   Metadata: {'type': 'current_activity', 'source': 'test_notebook'}


/tmp/ipykernel_1377587/440462837.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow()


In [11]:
# Check what memory tiers are available
print("🔍 Checking initialized memory tiers:\n")

tiers_status = {
    "Working Memory": smrti.working_memory,
    "Short-Term Memory": smrti.short_term_memory,
    "Long-Term Memory": smrti.long_term_memory,
    "Episodic Memory": smrti.episodic_memory,
    "Semantic Memory": getattr(smrti, 'semantic_memory', None)
}

available_tiers = []
missing_tiers = []

for name, tier in tiers_status.items():
    if tier is not None:
        print(f"   ✅ {name}: Available")
        available_tiers.append(name)
    else:
        print(f"   ❌ {name}: Not initialized")
        missing_tiers.append(name)

print(f"\n📊 Summary:")
print(f"   Available: {len(available_tiers)}/5 tiers")
print(f"   Missing: {len(missing_tiers)}/5 tiers")

if missing_tiers:
    print(f"\n⚠️ Missing tiers likely due to:")
    if "Long-Term Memory" in missing_tiers:
        print(f"   • ChromaDB not installed: pip install chromadb")
    if "Episodic Memory" in missing_tiers:
        print(f"   • PostgreSQL not configured or not running")
    if "Working Memory" in missing_tiers or "Short-Term Memory" in missing_tiers:
        print(f"   • Redis not running: docker run -d -p 6379:6379 redis")

print(f"\n🎯 We'll test the {len(available_tiers)} available tier(s)!")

🔍 Checking initialized memory tiers:

   ❌ Working Memory: Not initialized
   ❌ Short-Term Memory: Not initialized
   ❌ Long-Term Memory: Not initialized
   ❌ Episodic Memory: Not initialized
   ❌ Semantic Memory: Not initialized

📊 Summary:
   Available: 0/5 tiers
   Missing: 5/5 tiers

⚠️ Missing tiers likely due to:
   • ChromaDB not installed: pip install chromadb
   • PostgreSQL not configured or not running
   • Redis not running: docker run -d -p 6379:6379 redis

🎯 We'll test the 0 available tier(s)!


### 🔧 Quick Setup Commands

If tiers are missing, run these commands:

**Install ChromaDB** (for Long-Term Memory):
```bash
pip install chromadb
```

**Start Redis with Docker** (for Working & Short-Term Memory):
```bash
docker run -d --name smrti-redis -p 6379:6379 redis
```

**Start PostgreSQL with Docker** (for Episodic Memory - Optional):
```bash
docker run -d --name smrti-postgres \
  -e POSTGRES_PASSWORD=postgres \
  -e POSTGRES_DB=smrti \
  -p 5432:5432 postgres
```

After installing/starting services, restart the kernel and re-run from the beginning.

In [ ]:
# Store in working memory (direct tier access)
working_memory_id = await smrti.working_memory.store(
    record=working_memory_record,
    priority=0.8
)

print(f"✅ Stored in working memory")
print(f"   Memory ID: {working_memory_id}")

In [ ]:
# Retrieve from working memory
retrieved_working = await smrti.working_memory.retrieve(
    record_id=working_memory_id
)

print("✅ Retrieved from working memory:")
print(f"   Content: {retrieved_working.payload if retrieved_working else 'NOT FOUND'}")
print(f"   Metadata: {retrieved_working.metadata if retrieved_working else 'N/A'}")

In [ ]:
# Test working memory query
from smrti.schemas.models import MemoryQuery

working_search_query = MemoryQuery(
    user_id="user_123",
    query="testing",
    tenant="test",
    namespace="test_user_123",
    token_budget=4000
)

working_search_results = await smrti.working_memory.query(working_search_query)

print(f"🔍 Working memory search results: {len(working_search_results)} found")
for idx, result in enumerate(working_search_results[:3], 1):
    content = result.payload if isinstance(result.payload, str) else str(result.payload)
    print(f"   {idx}. {content[:50]}...")

## 💭 Test 2: Short-Term Memory (Redis with Sessions)

Short-term memory should:
- Store items for 24 hours (default TTL)
- Use Redis as backend
- Handle recent conversations/sessions

In [ ]:
# Create short-term memory items (simulating a conversation)
conversation = [
    "User asked about Smrti's architecture",
    "Assistant explained the four-tier memory system",
    "User requested to test working memory first",
    "Assistant created test cases for working memory"
]

short_term_ids = []

for turn, content in enumerate(conversation, 1):
    record = RecordEnvelope(
        tenant="test",
        namespace="test_user_123",
        user_id="user_123",
        tier="short_term",
        content_type="TEXT",
        payload=content,
        importance_score=0.6,
        metadata={
            "type": "conversation_turn",
            "turn": turn,
            "session_id": "test_session_001"
        },
        created_at=datetime.utcnow()
    )
    
    memory_id = await smrti.short_term_memory.store(
        record=record,
        importance=0.6,
        session_id="test_session_001"
    )
    
    short_term_ids.append(memory_id)
    print(f"✅ Turn {turn} stored: {memory_id}")

print(f"\n✅ Stored {len(short_term_ids)} conversation turns in short-term memory")

In [ ]:
# Retrieve a specific conversation turn
retrieved_turn = await smrti.short_term_memory.retrieve(
    record_id=short_term_ids[1]  # Second turn
)

print("✅ Retrieved conversation turn:")
print(f"   Content: {retrieved_turn.payload if retrieved_turn else 'NOT FOUND'}")
print(f"   Turn: {retrieved_turn.metadata.get('turn') if retrieved_turn else 'N/A'}")

In [ ]:
# Search short-term memory
short_term_query = MemoryQuery(
    user_id="user_123",
    query="architecture",
    tenant="test",
    namespace="test_user_123",
    token_budget=4000
)

short_term_search = await smrti.short_term_memory.query(short_term_query)

print(f"🔍 Short-term memory search: {len(short_term_search)} results")
for idx, result in enumerate(short_term_search, 1):
    content = result.payload if isinstance(result.payload, str) else str(result.payload)
    print(f"   {idx}. {content}")

## 📚 Test 3: Long-Term Memory (ChromaDB with Embeddings)

Long-term memory should:
- Store items permanently
- Use ChromaDB with vector embeddings
- Support semantic search
- Handle knowledge and facts

In [ ]:
# Create long-term knowledge items
knowledge_items = [
    {
        "content": "Smrti uses a four-tier memory architecture: Working, Short-Term, Long-Term, and Episodic memory",
        "category": "architecture",
        "importance": 0.9
    },
    {
        "content": "Working memory uses Redis with a 5-minute TTL for temporary, active information",
        "category": "working_memory",
        "importance": 0.8
    },
    {
        "content": "Long-term memory uses ChromaDB with sentence embeddings for semantic search capabilities",
        "category": "long_term_memory",
        "importance": 0.85
    },
    {
        "content": "Episodic memory stores temporal event sequences in PostgreSQL with timestamp-based queries",
        "category": "episodic_memory",
        "importance": 0.85
    },
    {
        "content": "The embedding provider uses SentenceTransformers with the all-MiniLM-L6-v2 model by default",
        "category": "embeddings",
        "importance": 0.75
    }
]

long_term_ids = []

for item in knowledge_items:
    record = RecordEnvelope(
        tenant="test",
        namespace="test_user_123",
        user_id="user_123",
        tier="long_term",
        content_type="FACT",
        payload=item["content"],
        importance_score=item["importance"],
        metadata={
            "type": "knowledge",
            "category": item["category"],
            "verified": True
        },
        created_at=datetime.utcnow()
    )
    
    memory_id = await smrti.long_term_memory.store(record=record)
    
    long_term_ids.append(memory_id)
    print(f"✅ Knowledge stored: {item['category']}")

print(f"\n✅ Stored {len(long_term_ids)} knowledge items in long-term memory")

In [ ]:
# Test semantic search in long-term memory
semantic_queries = [
    "What database does Smrti use for vector storage?",
    "How long does working memory keep data?",
    "What are the different memory tiers?",
    "Tell me about embeddings and semantic search"
]

print("🔍 Testing semantic search in long-term memory:\n")

for query_text in semantic_queries:
    print(f"Query: '{query_text}'")
    
    query = MemoryQuery(
        user_id="user_123",
        query=query_text,
        tenant="test",
        namespace="test_user_123",
        token_budget=4000
    )
    
    results = await smrti.long_term_memory.query(query)
    
    print(f"  Found {len(results)} results:")
    for idx, result in enumerate(results[:3], 1):
        score = result.relevance_score if hasattr(result, 'relevance_score') else 'N/A'
        content = result.payload if isinstance(result.payload, str) else str(result.payload)
        print(f"    {idx}. [Score: {score}] {content[:80]}...")
    print()

In [ ]:
# Retrieve specific long-term memory
retrieved_knowledge = await smrti.long_term_memory.retrieve(record_id=long_term_ids[0])

print("✅ Retrieved knowledge item:")
print(f"   Content: {retrieved_knowledge.payload if retrieved_knowledge else 'NOT FOUND'}")
print(f"   Category: {retrieved_knowledge.metadata.get('category') if retrieved_knowledge else 'N/A'}")

## 🎬 Test 4: Episodic Memory (PostgreSQL with Timestamps)

Episodic memory should:
- Store temporal event sequences
- Use PostgreSQL as backend
- Support time-based queries
- Handle event causality

In [ ]:
# Create episodic events (simulating a user journey)
base_time = datetime.utcnow()

episodes = [
    {
        "content": "User opened Smrti documentation for the first time",
        "timestamp": base_time - timedelta(hours=2),
        "event_type": "documentation_access",
        "importance": 0.6
    },
    {
        "content": "User asked about the difference between memory tiers",
        "timestamp": base_time - timedelta(hours=1, minutes=45),
        "event_type": "question",
        "importance": 0.7
    },
    {
        "content": "User installed required dependencies (chromadb, redis, asyncpg)",
        "timestamp": base_time - timedelta(hours=1, minutes=30),
        "event_type": "setup",
        "importance": 0.8
    },
    {
        "content": "User successfully initialized Smrti for the first time",
        "timestamp": base_time - timedelta(hours=1),
        "event_type": "milestone",
        "importance": 0.9
    },
    {
        "content": "User started testing working memory functionality",
        "timestamp": base_time - timedelta(minutes=30),
        "event_type": "testing",
        "importance": 0.75
    },
    {
        "content": "User tested semantic search in long-term memory",
        "timestamp": base_time - timedelta(minutes=10),
        "event_type": "testing",
        "importance": 0.8
    },
    {
        "content": "User is now testing episodic memory",
        "timestamp": base_time,
        "event_type": "testing",
        "importance": 0.85
    }
]

episodic_ids = []

for episode in episodes:
    record = RecordEnvelope(
        tenant="test",
        namespace="test_user_123",
        user_id="user_123",
        tier="episodic",
        content_type="EVENT",
        payload=episode["content"],
        importance_score=episode["importance"],
        metadata={
            "type": "episode",
            "event_type": episode["event_type"],
            "timestamp": episode["timestamp"].isoformat()
        },
        created_at=episode["timestamp"]
    )
    
    memory_id = await smrti.episodic_memory.store(record=record)
    
    episodic_ids.append(memory_id)
    print(f"✅ Episode stored: {episode['event_type']} - {episode['content'][:50]}...")

print(f"\n✅ Stored {len(episodic_ids)} episodes in episodic memory")

In [ ]:
# Search episodic memory
episodic_query = MemoryQuery(
    user_id="user_123",
    query="testing",
    tenant="test",
    namespace="test_user_123",
    token_budget=4000
)

episodic_search = await smrti.episodic_memory.query(episodic_query)

print(f"🔍 Episodic memory search: {len(episodic_search)} results")
print("\nTimeline of testing events:")
for idx, result in enumerate(episodic_search, 1):
    timestamp = result.metadata.get('timestamp', 'Unknown')
    event_type = result.metadata.get('event_type', 'Unknown')
    content = result.payload if isinstance(result.payload, str) else str(result.payload)
    print(f"   {idx}. [{event_type}] {content}")
    print(f"      Time: {timestamp}")

In [ ]:
# Retrieve specific episode
retrieved_episode = await smrti.episodic_memory.retrieve(record_id=episodic_ids[3])  # The milestone event

print("✅ Retrieved milestone episode:")
print(f"   Content: {retrieved_episode.payload if retrieved_episode else 'NOT FOUND'}")
print(f"   Event Type: {retrieved_episode.metadata.get('event_type') if retrieved_episode else 'N/A'}")
print(f"   Importance: {retrieved_episode.importance_score if retrieved_episode else 'N/A'}")

## 🔄 Test 5: Cross-Tier Operations

Test how different tiers interact and whether we can query across tiers

In [ ]:
# Get stats from all tiers
print("📊 Memory Tier Statistics:\n")

tiers = [
    ("Working Memory", smrti.working_memory),
    ("Short-Term Memory", smrti.short_term_memory),
    ("Long-Term Memory", smrti.long_term_memory),
    ("Episodic Memory", smrti.episodic_memory)
]

for tier_name, tier in tiers:
    try:
        # Try to get stats if available
        if hasattr(tier, 'get_stats'):
            stats = await tier.get_stats()
            print(f"{tier_name}:")
            print(f"  Stats: {stats}")
        else:
            print(f"{tier_name}: Stats method not available")
    except Exception as e:
        print(f"{tier_name}: Error - {e}")
    print()

In [ ]:
# Test registry access
print("🗂️ Registry Information:\n")

print(f"Registered Memory Tiers: {list(smrti.registry.memory_tiers.keys())}")
print(f"Registered Tier Stores: {list(smrti.registry.tier_stores.keys())}")
print(f"Registered Embedding Providers: {list(smrti.registry.embedding_providers.keys())}")

# Check if we can access adapters
print(f"\nStorage Adapters: {list(smrti.registry.storage_adapters.keys()) if hasattr(smrti.registry, 'storage_adapters') else 'N/A'}")
print(f"Vector Adapters: {list(smrti.registry.vector_adapters.keys()) if hasattr(smrti.registry, 'vector_adapters') else 'N/A'}")

## 🧪 Test 6: Edge Cases & Error Handling

In [ ]:
# Test 1: Retrieve non-existent memory
print("Test 1: Retrieve non-existent memory")
try:
    fake_id = "non_existent_memory_12345"
    result = await smrti.working_memory.retrieve(record_id=fake_id)
    print(f"   Result: {result}")
    print(f"   ✅ Handled gracefully (returned None or empty)" if result is None else "   ⚠️ Unexpected result")
except Exception as e:
    print(f"   ❌ Exception: {type(e).__name__}: {e}")

print()

In [ ]:
# Test 2: Empty query search
print("Test 2: Empty query search")
try:
    empty_query = MemoryQuery(
        user_id="user_123",
        query="",
        tenant="test",
        namespace="test_user_123",
        token_budget=4000
    )
    results = await smrti.long_term_memory.query(empty_query)
    print(f"   Results: {len(results)} items")
    print(f"   ✅ Handled gracefully")
except Exception as e:
    print(f"   ❌ Exception: {type(e).__name__}: {e}")

print()

In [ ]:
# Test 3: Very long content
print("Test 3: Very long content (10,000 characters)")
try:
    long_content = "This is a test of very long content. " * 250  # ~10k characters
    long_record = RecordEnvelope(
        tenant="test",
        namespace="test_user_123",
        user_id="user_123",
        tier="long_term",
        content_type="TEXT",
        payload=long_content,
        metadata={"type": "stress_test", "length": len(long_content)},
        created_at=datetime.utcnow()
    )
    
    memory_id = await smrti.long_term_memory.store(record=long_record)
    
    print(f"   ✅ Stored successfully: {memory_id}")
    print(f"   Content length: {len(long_content)} characters")
except Exception as e:
    print(f"   ❌ Exception: {type(e).__name__}: {e}")

print()

In [ ]:
# Test 4: Special characters and unicode
print("Test 4: Special characters and unicode")
try:
    special_content = "Testing special chars: @#$%^&*() 中文 العربية 🧠💭🎯 \n\t\r"
    special_record = RecordEnvelope(
        tenant="test",
        namespace="test_user_123",
        user_id="user_123",
        tier="working",
        content_type="TEXT",
        payload=special_content,
        metadata={"type": "unicode_test"},
        created_at=datetime.utcnow()
    )
    
    memory_id = await smrti.working_memory.store(record=special_record)
    
    retrieved = await smrti.working_memory.retrieve(record_id=memory_id)
    
    print(f"   ✅ Stored and retrieved successfully")
    print(f"   Original: {special_content}")
    print(f"   Retrieved: {retrieved.payload if retrieved else 'NONE'}")
    print(f"   Match: {retrieved.payload == special_content if retrieved else False}")
except Exception as e:
    print(f"   ❌ Exception: {type(e).__name__}: {e}")

print()

In [ ]:
# Test 5: Storing duplicate content
print("Test 5: Storing duplicate content")
try:
    duplicate_content = "This is duplicate content for testing"
    
    ids = []
    for i in range(3):
        record = RecordEnvelope(
            tenant="test",
            namespace="test_user_123",
            user_id="user_123",
            tier="short_term",
            content_type="TEXT",
            payload=duplicate_content,
            metadata={"type": "duplicate_test", "instance": i+1},
            created_at=datetime.utcnow()
        )
        
        memory_id = await smrti.short_term_memory.store(record=record)
        ids.append(memory_id)
    
    print(f"   ✅ Stored 3 duplicate items")
    print(f"   IDs: {ids}")
    print(f"   All unique: {len(set(ids)) == 3}")
except Exception as e:
    print(f"   ❌ Exception: {type(e).__name__}: {e}")

print()

## 📊 Test Summary & Health Check

In [ ]:
# Final system stats
print("🏁 FINAL SYSTEM STATUS\n")
print("=" * 60)

final_stats = await smrti.get_system_stats()
print("\n📊 System Statistics:")
print(json.dumps(final_stats, indent=2))

print("\n💚 Health Status:")
health = smrti.get_health_status()
print(f"   Status: {health}")

print("\n" + "=" * 60)

In [ ]:
# Test Summary
print("\n✅ TESTS COMPLETED\n")
print("Tested Components:")
print("  ✓ Working Memory (Redis with TTL)")
print("  ✓ Short-Term Memory (Redis with sessions)")
print("  ✓ Long-Term Memory (ChromaDB with embeddings)")
print("  ✓ Episodic Memory (PostgreSQL with timestamps)")
print("  ✓ Semantic search capabilities")
print("  ✓ Memory retrieval")
print("  ✓ Registry access")
print("  ✓ Edge cases and error handling")
print("\nNext Steps:")
print("  1. Review LEARNING.md for detailed findings")
print("  2. Implement high-level API (store_memory, search_memories)")
print("  3. Add content validation per tier")
print("  4. Build use case showcases")

## 🧹 Cleanup

In [ ]:
# Shutdown Smrti
await smrti.shutdown()
print("✅ Smrti shutdown complete")